# SwinIR ×3 — GAN 없이 Transformer 로

흐린 위성사진(10 m) → 3배 선명하게(3.33 m).

앞의 세 모델과 달리 **판별자가 없다.** 손실은 L1 하나뿐이라 붕괴할 것도 없다.

| | 구조 | 손실 |
|---|---|---|
| EDSR | CNN (residual) | L1 |
| SRGAN | CNN + GAN | MSE + VGG + 적대적 |
| ESRGAN | CNN (RRDB) + GAN | L1 + VGG + RaGAN |
| **SwinIR** | **Transformer (Swin)** | **L1** |

## 1. 데이터

In [ ]:
import sys, urllib.request
!pip install -q timm

LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'swinir_arch.py', 'swinir_models.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)
    sys.modules.pop(m[:-3], None)     # 이미 불러온 옛 모듈이 남아 있으면 비운다

from sr_utils import *

show_data()        # validation 2패치 + test 2구역

## 2. 훈련

**코드가 도는지 확인하는 용도다.** 16장으로 1 epoch 만 돌린다.
아래 결과는 전체 데이터로 학습해둔 가중치를 쓴다.

판별자가 없어서 루프가 짧다. 예측하고, L1 손실을 재고, 갱신하는 것이 전부다.

SwinIR 은 어텐션 기반이라 메모리를 많이 쓴다. batch 1 로 둔다.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from swinir_models import build_swinir

# SwinIR 은 어텐션이라 입력이 커지면 메모리가 급격히 는다. batch 1 로 둔다 (약 6 GB).
N_TRAIN, EPOCHS, BATCH = 16, 1, 1
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = zip(*[pair('training', s) for s in list_split('training')[:N_TRAIN]])
to_t = lambda a: torch.from_numpy(np.stack(a).transpose(0, 3, 1, 2)).float() / 255
loader = DataLoader(TensorDataset(to_t(lo), to_t(hi)), batch_size=BATCH, shuffle=True)

net = build_swinir(3).to(dev).train()
opt = torch.optim.Adam(net.parameters(), 2e-4, betas=(0.9, 0.99))
crit = nn.L1Loss()

for ep in range(1, EPOCHS + 1):
    tot = 0.0
    for x, y in loader:
        loss = crit(net(x.to(dev)), y.to(dev))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        tot += loss.item()
    print(f'epoch {ep}/{EPOCHS}   L1 {tot/len(loader):.5f}')

## 3. 결과

In [ ]:
MODEL = f'{BASE}/models/06_swinir_x3'

from swinir_models import load_swinir

net = load_swinir(fetch(f'{MODEL}/checkpoints/swinir_x3.pth', 'swinir_x3.pth'))
print(f'SwinIR classical  {sum(p.numel() for p in net.parameters())/1e6:.2f}M')

@torch.no_grad()
def upscale(lr):
    t = torch.from_numpy(lr.transpose(2, 0, 1)).float()[None].to(next(net.parameters()).device) / 255
    return (net(t).clamp(0, 1)[0].cpu().numpy().transpose(1, 2, 0) * 255).round().astype('uint8')

show_results(upscale, 'SwinIR', center=[(137, 236), (313, 346)])

## 4. 평가

In [ ]:
rows = compare(upscale, label='SwinIR')

## 5. 최종 테스트 — 인천

정답이 없는 실제 Sentinel-2 촬영본이다. 점수는 못 내고 눈으로 확인한다.

In [ ]:
show_test(upscale, 'SwinIR', center=(800, 800), size=70)